In [25]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

### Load Config

In [27]:
from config import dir_config, main_config

processed_dir = Path(dir_config.data.processed)

mds_updrs_conf = main_config.MDS_UPDRS
metadata = pd.read_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), encoding="latin1", index_col=None)

In [28]:
def get_valid_vars(item_list, df_columns, label):
    valid_vars = []
    for var in item_list:
        assert var in df_columns, f"Variable {var} not found in DataFrame columns for {label}"
        valid_vars.append(var)
    return valid_vars


# Helper to safely convert to numeric
def to_numeric_df(df):
    return df.apply(pd.to_numeric, errors="coerce")

In [29]:
all_subjects = metadata["subject_id"].unique()
stanford_subjects = metadata[metadata["experiment_site"] == "Stanford"]["subject_id"].unique()
ucla_subjects = metadata[metadata["experiment_site"] == "UCLA"]["subject_id"].unique()
case_western_subjects = metadata[metadata["experiment_site"] == "Case_Western"]["subject_id"].unique()
harvard_subjects = metadata[metadata["experiment_site"] == "Harvard"]["subject_id"].unique()

# Subjects that have at least one treatment session are PD; the rest are HC
pd_subjects = metadata[metadata["treatment"].notna()]["subject_id"].unique()
hc_subjects = [s for s in all_subjects if s not in pd_subjects]

new_pd_subjects = list(set(stanford_subjects) | (set(harvard_subjects) & set(pd_subjects)))
new_pd_subjects_idx = metadata.index[metadata["subject_id"].isin(new_pd_subjects)]

print(f"Total subjects: {len(all_subjects)} (PD: {len(pd_subjects)}, HC: {len(hc_subjects)})")
print(f"Stanford: {len(stanford_subjects)}, UCLA: {len(ucla_subjects)}, Case Western: {len(case_western_subjects)}, Harvard: {len(harvard_subjects)}")
print(f"New-cohort PD subjects: {len(new_pd_subjects)}")
print(f"New-cohort healthy subjects: {len(hc_subjects)}")

Total subjects: 59 (PD: 41, HC: 18)
Stanford: 16, UCLA: 25, Case Western: 4, Harvard: 14
New-cohort PD subjects: 22
New-cohort healthy subjects: 18


In [30]:
relevant_UPDRS_vars = get_valid_vars(mds_updrs_conf.UPDRS_ITEMS, metadata.columns, "UPDRS")
relevant_tremor_vars = get_valid_vars(mds_updrs_conf.TREMOR_ITEMS, metadata.columns, "TREMOR")
relevant_bradykinesia_vars = get_valid_vars(mds_updrs_conf.BRADYKINESIA_ITEMS, metadata.columns, "BRADYKINESIA")
relevant_pigd_vars = get_valid_vars(mds_updrs_conf.PIGD_ITEMS, metadata.columns, "PIGD")

In [31]:
updrs_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_UPDRS_vars])
tremor_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_tremor_vars])
brady_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_bradykinesia_vars])
pigd_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_pigd_vars])

In [32]:
# Compute scores
updrs_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_UPDRS_vars])
tremor_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_tremor_vars])
brady_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_bradykinesia_vars])
pigd_numeric = to_numeric_df(metadata.loc[new_pd_subjects_idx, relevant_pigd_vars])

metadata.loc[new_pd_subjects_idx, "UPDRS"] = np.nansum(updrs_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "tremor_score"] = np.nanmean(tremor_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "bradykinesia_score"] = np.nanmean(brady_numeric, axis=1)
metadata.loc[new_pd_subjects_idx, "pigd_score"] = np.nanmean(pigd_numeric, axis=1)

# Initialize ratio columns for all rows before computing
metadata["trem_by_brady"] = np.nan
metadata["trem_by_pigd"] = np.nan

# Compute ratios with division safety
with np.errstate(divide="ignore", invalid="ignore"):
    trem_by_brady = metadata.loc[new_pd_subjects_idx, "tremor_score"] / metadata.loc[new_pd_subjects_idx, "bradykinesia_score"]
    trem_by_pigd = metadata.loc[new_pd_subjects_idx, "tremor_score"] / metadata.loc[new_pd_subjects_idx, "pigd_score"]

# Assign results
metadata.loc[new_pd_subjects_idx, "trem_by_brady"] = trem_by_brady.replace([np.inf, -np.inf], np.nan)
metadata.loc[new_pd_subjects_idx, "trem_by_pigd"] = trem_by_pigd.replace([np.inf, -np.inf], np.nan)

In [33]:
def get_label(value, thresholds):
    if pd.isna(value):
        return np.nan
    for label, (low, high) in thresholds.items():
        if low <= value <= high:
            return label
    print(f"Warning: value {value} did not match any threshold.")
    return "unknown"


for subject in new_pd_subjects:
    subject_indices = metadata.index[metadata["subject_id"] == subject]
    subject_data = metadata.loc[subject_indices]

    off_rows = subject_data[subject_data["treatment"] == "OFF"]

    if off_rows.empty:
        print(f"No OFF-treatment data for subject {subject}. Skipping...")
        continue

    # Apply get_label to OFF-treatment row(s)
    trem_vs_brady_type = off_rows["trem_by_brady"].apply(lambda x: get_label(x, mds_updrs_conf.TREM_VS_BRADY)).iloc[0]
    pigd_vs_tremor_type = off_rows["trem_by_pigd"].apply(lambda x: get_label(x, mds_updrs_conf.TREM_VS_PIDG)).iloc[0]

    metadata.loc[subject_indices, "trem_vs_brady_type"] = trem_vs_brady_type
    metadata.loc[subject_indices, "pigd_vs_tremor_type"] = pigd_vs_tremor_type

# Now create composite columns
metadata["trem_vs_non-trem"] = np.where(
    (metadata["trem_vs_brady_type"] == "tremor") | (metadata["pigd_vs_tremor_type"] == "tremor"),
    "tremor",
    "non-tremor",
)

metadata["pigd_vs_non-pigd"] = np.where(metadata["pigd_vs_tremor_type"] == "pigd", "pigd", "non-pigd")

metadata.loc[metadata["is_pd"] == 0, "trem_vs_non-trem"] = np.nan
metadata.loc[metadata["is_pd"] == 0, "pigd_vs_non-pigd"] = np.nan

In [34]:
# Ensure improvement columns exist
metadata["UPDRS_improvement"] = np.nan
metadata["tremor_improvement"] = np.nan
metadata["bradykinesia_improvement"] = np.nan

for sub in metadata["subject_id"].unique():
    sub_data = metadata[metadata["subject_id"] == sub]
    if sub_data["treatment"].nunique() < 2:
        continue  # skip if both OFF and ON not present

    try:
        off_row = sub_data[sub_data["treatment"] == "OFF"].iloc[0]
        on_row = sub_data[sub_data["treatment"] == "ON"].iloc[0]

        def percent_improvement(off, on):
            return np.nan if off == 0 else (off - on) * 100 / off

        ui = percent_improvement(off_row["UPDRS"], on_row["UPDRS"])
        ti = percent_improvement(off_row["tremor_score"], on_row["tremor_score"])
        bi = percent_improvement(off_row["bradykinesia_score"], on_row["bradykinesia_score"])

        metadata.loc[sub_data.index, "UPDRS_improvement"] = ui
        metadata.loc[sub_data.index, "tremor_improvement"] = ti
        metadata.loc[sub_data.index, "bradykinesia_improvement"] = bi

    except (IndexError, KeyError, TypeError):
        continue  # skip subjects with missing or malformed data

In [35]:
metadata["Were_Diskinesias_Present"] = metadata["Were_Diskinesias_Present"].map({"yes": 1, "no": 0})
metadata["Did_these_Movements_Interfere_with_Ratings"] = metadata["Did_these_Movements_Interfere_with_Ratings"].map({"yes": 1, "no": 0})

In [36]:
metadata.to_csv(Path(processed_dir, "processed_metadata_all_data_accu_60.csv"), index=False)